In [2]:
import torch
import torch.nn as nn
import onnx
import os
import json


symbols = [
    "GOOGL","AVGO","MSFT","META","NVDA","AAPL","TSLA",
]
# ── Model Definition ──────────────────────────────────────────────────────────
class StockPctChangeBiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2, output_size=1):
        super().__init__()
        self.lstm      = nn.LSTM(input_size, hidden_size, num_layers,
                                 batch_first=True, bidirectional=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.attention      = nn.Linear(hidden_size * 2, 1)
        self.pos_bias       = nn.Parameter(torch.zeros(1))
        self.ln             = nn.LayerNorm(hidden_size * 2)
        self.dropout        = nn.Dropout(dropout)
        self.fc             = nn.Linear(hidden_size * 2, output_size)
        self.skip_fc        = nn.Linear(hidden_size * 2, output_size)
        self.output_blend   = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        lstm_out, _  = self.lstm(x)
        seq_len      = lstm_out.size(1)
        pos_weights  = torch.linspace(0, 1, seq_len, device=x.device).unsqueeze(-1)
        attn_raw     = self.attention(lstm_out) + self.pos_bias * pos_weights
        attn_weights = torch.softmax(attn_raw, dim=1)
        context      = (attn_weights * lstm_out).sum(dim=1)
        out_attn     = self.fc(self.dropout(self.ln(context)))
        last_hidden  = lstm_out[:, -1, :]
        out_skip     = self.skip_fc(last_hidden)
        alpha        = torch.sigmoid(self.output_blend)
        return alpha * out_attn + (1 - alpha) * out_skip


def infer_hyperparams(state_dict: dict, symbol: str = None) -> dict:
    """
    Build params from checkpoint tensor shapes (authoritative — they ARE the trained
    weights), then CROSS-CHECK against the saved Optuna best_params.json.

    Shapes:
      lstm.weight_ih_l0 (4*hidden, input_size) -> hidden_size, input_size
      count of lstm.weight_ih_lN (no _reverse) -> num_layers
      fc.weight (output_size, hidden*2)        -> output_size
    JSON is loaded ONLY to verify; if hidden_size/num_layers disagree with the
    checkpoint we raise, so a stale .pt or JSON can't silently mis-build the model.
    """
    ih_l0       = state_dict["lstm.weight_ih_l0"]
    hidden_size = ih_l0.shape[0] // 4
    input_size  = ih_l0.shape[1]
    num_layers  = sum(1 for k in state_dict
                      if k.startswith("lstm.weight_ih_l") and "_reverse" not in k)
    output_size = state_dict["fc.weight"].shape[0]

    hp = {"input_size": input_size, "hidden_size": hidden_size,
          "num_layers": num_layers, "output_size": output_size,
          "seq_len": None, "json_verified": False}

    if symbol is not None:
        jpath = f"models/{symbol}/{symbol}_best_params.json"
        if os.path.exists(jpath):
            with open(jpath) as f:
                bp = json.load(f).get("best_params", {})
            for key in ("hidden_size", "num_layers"):
                if key in bp and bp[key] != hp[key]:
                    raise ValueError(
                        f"[{symbol}] {key} mismatch: checkpoint={hp[key]} vs "
                        f"best_params.json={bp[key]} -- stale .pt or JSON, refusing to convert."
                    )
            hp["seq_len"]       = bp.get("seq_len")
            hp["json_verified"] = True
        else:
            print(f"[{symbol}] note: no best_params.json found -> using checkpoint shapes only")
    return hp


def convert(symbol: str):
    pt_path   = f"models/{symbol}/{symbol}_best_bilstm.pt"
    onnx_dir  = f"models/{symbol}"
    onnx_path = f"{onnx_dir}/{symbol}_bilstm.onnx"

    if not os.path.exists(pt_path):
        print(f"[SKIP] {symbol}: file not found -> {pt_path}")
        return

    print(f"\n[{symbol}] Loading {pt_path} ...")
    state_dict = torch.load(pt_path, map_location="cpu", weights_only=True)
    if "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]
    elif "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    hp = infer_hyperparams(state_dict, symbol)
    print(f"[{symbol}] params -> input_size={hp['input_size']}, hidden_size={hp['hidden_size']}, "
          f"num_layers={hp['num_layers']}, output_size={hp['output_size']}, "
          f"seq_len={hp['seq_len']}, json_verified={hp['json_verified']}")

    model = StockPctChangeBiLSTMAttention(
        input_size  = hp["input_size"],
        hidden_size = hp["hidden_size"],
        num_layers  = hp["num_layers"],
        output_size = hp["output_size"],
    )
    model.load_state_dict(state_dict)   # raises if shapes mismatch -> extra safety
    model.eval()

    seq = hp["seq_len"] or 20           # dynamic axis anyway; use real seq_len for the trace
    dummy_input = torch.randn(1, seq, hp["input_size"])

    os.makedirs(onnx_dir, exist_ok=True)
    torch.onnx.export(
        model, dummy_input, onnx_path,
        opset_version=17,
        input_names=["input"], output_names=["output"],
        dynamic_axes={"input": {0: "batch_size", 1: "seq_len"},
                      "output": {0: "batch_size"}},
        do_constant_folding=True,
    )
    onnx.checker.check_model(onnx.load(onnx_path))
    print(f"[{symbol}] OK Saved & validated -> {onnx_path}")


# ── Run all conversions ───────────────────────────────────────────────────────
if __name__ == "__main__":
    for sym in symbols:
        try:
            convert(sym)
        except Exception as e:
            print(f"[{sym}] X Error: {e}")



[GOOGL] Loading models/GOOGL/GOOGL_best_bilstm.pt ...
[GOOGL] params -> input_size=27, hidden_size=64, num_layers=2, output_size=1, seq_len=20, json_verified=True


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\torch\onnx\symbolic_opset9.py:4279: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


[GOOGL] OK Saved & validated -> models/GOOGL/GOOGL_bilstm.onnx

[AVGO] Loading models/AVGO/AVGO_best_bilstm.pt ...
[AVGO] params -> input_size=27, hidden_size=64, num_layers=2, output_size=1, seq_len=10, json_verified=True
[AVGO] OK Saved & validated -> models/AVGO/AVGO_bilstm.onnx

[MSFT] Loading models/MSFT/MSFT_best_bilstm.pt ...
[MSFT] params -> input_size=27, hidden_size=128, num_layers=2, output_size=1, seq_len=10, json_verified=True
[MSFT] OK Saved & validated -> models/MSFT/MSFT_bilstm.onnx

[META] Loading models/META/META_best_bilstm.pt ...
[META] params -> input_size=27, hidden_size=64, num_layers=2, output_size=1, seq_len=20, json_verified=True
[META] OK Saved & validated -> models/META/META_bilstm.onnx

[NVDA] Loading models/NVDA/NVDA_best_bilstm.pt ...
[NVDA] params -> input_size=27, hidden_size=64, num_layers=2, output_size=1, seq_len=20, json_verified=True
[NVDA] OK Saved & validated -> models/NVDA/NVDA_bilstm.onnx

[AAPL] Loading models/AAPL/AAPL_best_bilstm.pt ...
[AAP